# Standalone Colab: multilingual overlapping-speech experiment

This notebook is self-contained. It does **not** clone the Dhvani repository, import `src`, read `requirements.txt`, or depend on local project files.

**Data loading:** You manually upload one parquet shard per language to `/content/`. The notebook reads audio bytes directly from the parquet `audio_filepath` column, extracts one sample per language, creates controlled overlapping mixtures, runs pyannote diarization, transcribes with the IndicWav2Vec hosted endpoints (or Sarvam fallback), sends the raw transcript to Gemini for grounded cleanup, and creates result tables/plots.

**Supported languages (IndicWav2Vec):** Hindi (hi), Marathi (mr), Tamil (ta), Telugu (te).


In [ ]:
# Enter secrets only through hidden prompts; do not print or save them.
import os
from getpass import getpass

# HF_TOKEN is still needed for the gated pyannote diarization model.
HF_TOKEN = getpass("Hugging Face token (needed for pyannote): ").strip()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    print("WARNING: No HF_TOKEN provided. Pyannote diarization will fail if the model is gated.")

for name in ("SARVAM_API_KEY", "GEMINI_API_KEY"):
    value = getpass(f"{name} (optional; press Enter to skip): ").strip()
    if value:
        os.environ[name] = value
print("Secrets loaded without displaying their values.")


## Install Colab dependencies

This is intentionally a direct install list. No local file is required.

In [ ]:
# Install dependencies.
# Note: Google Colab may show a dependency conflict warning for pre-installed google-adk (opentelemetry).
# This is a harmless pre-existing Colab environment conflict.
%pip install -q soundfile librosa pandas numpy matplotlib seaborn requests scikit-learn jiwer indic-nlp-library google-genai pyannote.audio speechbrain pyarrow


In [ ]:
from __future__ import annotations

import base64
import io
import json
import random
import time
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import soundfile as sf

ROOT = Path("/content/standalone_indic_experiment")
DATA = ROOT / "data"
MIX = ROOT / "mixtures"
RESULTS = ROOT / "results"
for path in (DATA, MIX, RESULTS):
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
rng = np.random.default_rng(SEED)
SAMPLE_RATE = 16_000
MAX_SECONDS = 20
MAX_DATASET_BYTES = 5 * 1024**3

# Only the 4 languages that IndicWav2Vec can transcribe (audio -> text)
INDICWAV2VEC_LANGUAGES = {
    "hi": "Hindi",
    "mr": "Marathi",
    "ta": "Tamil",
    "te": "Telugu",
}
LANGUAGES = {v: k for k, v in INDICWAV2VEC_LANGUAGES.items()}

# Manually uploaded parquet shards — one per language config from IndicVoices.
# Upload these files to /content/ before running.
PARQUET_FILES = [
    "/content/train-00000-of-00072.parquet",
    "/content/train-00000-of-00061.parquet",
    "/content/train-00000-of-00118.parquet",
    "/content/train-00000-of-00082.parquet",
]

print(f"Target languages for IndicWav2Vec: {list(LANGUAGES.keys())}")
print(f"Standalone workspace: {ROOT}")
print(f"Parquet files to load: {len(PARQUET_FILES)}")


## Load one clip per language from uploaded parquet files

Each parquet file is one shard from `ai4bharat/IndicVoices`. The `lang` column identifies the language and `audio_filepath` contains the raw FLAC bytes. We extract one good sample per IndicWav2Vec-supported language (hi, mr, ta, te), decode the audio, resample to 16 kHz, and cap at 20 seconds.


In [ ]:
def decode_audio_bytes(raw_bytes):
    """Decode raw audio bytes (FLAC/WAV) to a float32 numpy array at SAMPLE_RATE."""
    audio, source_rate = sf.read(io.BytesIO(raw_bytes), dtype="float32")
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if source_rate != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=source_rate, target_sr=SAMPLE_RATE)
    audio = audio[: MAX_SECONDS * SAMPLE_RATE].astype(np.float32)
    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    return audio / peak if peak > 1.0 else audio

download_rows = []
download_notes = []
collected_codes = set()

for pq_path in PARQUET_FILES:
    pq_file = Path(pq_path)
    if not pq_file.is_file():
        download_notes.append(f"File not found: {pq_path}")
        continue

    print(f"Reading {pq_file.name}...")
    df = pd.read_parquet(pq_file)
    print(f"  Shape: {df.shape}, languages present: {df['lang'].unique().tolist()}")

    # Filter to only IndicWav2Vec-supported languages
    for lang_code in df["lang"].unique():
        if lang_code not in INDICWAV2VEC_LANGUAGES:
            continue
        if lang_code in collected_codes:
            continue  # already have a sample for this language

        lang_name = INDICWAV2VEC_LANGUAGES[lang_code]
        lang_df = df[df["lang"] == lang_code]

        # Pick one sample with duration >= 2s and <= 20s
        candidates = lang_df[(lang_df["duration"] >= 2.0) & (lang_df["duration"] <= MAX_SECONDS)]
        if candidates.empty:
            candidates = lang_df[lang_df["duration"] >= 1.0]
        if candidates.empty:
            download_notes.append(f"{lang_name} ({lang_code}): no suitable sample in {pq_file.name}")
            continue

        row = candidates.iloc[0]
        try:
            # audio_filepath column contains a dict with 'bytes' key
            audio_data = row["audio_filepath"]
            if isinstance(audio_data, dict):
                raw_bytes = audio_data["bytes"]
            elif isinstance(audio_data, bytes):
                raw_bytes = audio_data
            else:
                download_notes.append(f"{lang_name}: unexpected audio_filepath type: {type(audio_data)}")
                continue

            audio = decode_audio_bytes(raw_bytes)
            if len(audio) < SAMPLE_RATE:
                download_notes.append(f"{lang_name}: decoded audio too short ({len(audio)} samples)")
                continue

            # Get transcript text
            text = ""
            for col in ("normalized", "text", "verbatim"):
                val = row.get(col)
                if val and isinstance(val, str) and val.strip():
                    text = val.strip()
                    break

            wav_path = DATA / f"{lang_code}.wav"
            sf.write(wav_path, audio, SAMPLE_RATE, subtype="PCM_16")

            download_rows.append({
                "language": lang_name,
                "code": lang_code,
                "audio": str(wav_path),
                "text": text,
                "duration_s": round(len(audio) / SAMPLE_RATE, 3),
                "source": pq_file.name,
            })
            collected_codes.add(lang_code)
            print(f"  [OK] {lang_name} ({lang_code}): {round(len(audio)/SAMPLE_RATE, 2)}s from {pq_file.name}")

        except Exception as exc:
            download_notes.append(f"{lang_name} ({lang_code}): {type(exc).__name__}: {str(exc)[:160]}")

download_manifest = pd.DataFrame(download_rows)
download_manifest.to_csv(DATA / "manifest.csv", index=False)

if not download_manifest.empty:
    display(download_manifest[["language", "code", "duration_s", "source"]])
else:
    print("No samples collected!")

if download_notes:
    print(f"\nNotes ({len(download_notes)}):")
    for note in download_notes:
        print(f"  * {note}")

# Show which IndicWav2Vec languages we're still missing
missing = set(INDICWAV2VEC_LANGUAGES.keys()) - collected_codes
if missing:
    print(f"\nWARNING: Missing languages: {[INDICWAV2VEC_LANGUAGES[c] for c in missing]}")
    print("These languages were not found in any uploaded parquet file.")


In [ ]:
def folder_bytes(path):
    return sum(item.stat().st_size for item in Path(path).rglob("*") if item.is_file())

used = folder_bytes(ROOT)
print(f"Stored dataset bytes: {used / 1024**2:.2f} MiB")
assert used <= MAX_DATASET_BYTES, "The 5 GB dataset limit was exceeded. Reduce the slice."

if len(download_rows) < 2:
    if download_notes:
        print("\nDownload failure diagnostics:")
        for note in download_notes:
            print(" -", note)
    raise RuntimeError(
        f"At least two language samples are required to create an overlap mixture, "
        f"but collected {len(download_rows)}. "
        f"Make sure you uploaded parquet files containing hi/mr/ta/te data to /content/."
    )
print(f"Successfully collected {len(download_rows)} samples: "
      f"{[r['language'] for r in download_rows]}")


## Create overlap mixtures and references

Each mixture has two clean reference tracks, controlled overlap, a target SIR, gain variation, and background noise. The references are used for separation metrics.

In [ ]:
mixture_rows = []
for index in range(min(12, max(1, len(download_rows) * 2))):
    left, right = random.sample(download_rows, 2)
    first, _ = librosa.load(left["audio"], sr=SAMPLE_RATE, mono=True, duration=12)
    second, _ = librosa.load(right["audio"], sr=SAMPLE_RATE, mono=True, duration=12)
    length = min(len(first), len(second))
    first, second = first[:length], second[:length]
    overlap = random.choice((0.25, 0.5, 0.75, 1.0))
    offset = int((1.0 - overlap) * length)
    source_1 = np.pad(first, (0, max(0, offset + len(second) - len(first))))
    source_2 = np.pad(second, (offset, 0))
    total = max(len(source_1), len(source_2))
    source_1 = np.pad(source_1, (0, total - len(source_1)))
    source_2 = np.pad(source_2, (0, total - len(source_2)))
    sir_db = random.choice((0.0, 5.0))
    rms_1 = np.sqrt(np.mean(source_1**2) + 1e-8)
    rms_2 = np.sqrt(np.mean(source_2**2) + 1e-8)
    source_2 *= rms_1 / (10 ** (sir_db / 20) * rms_2)
    mixture = source_1 + source_2 + rng.normal(0, 0.0025, total).astype(np.float32)
    peak = max(float(np.max(np.abs(mixture))), 1.0)
    mixture_id = f"mix_{index:03d}"
    for name, audio in (("mix", mixture), ("source_1", source_1), ("source_2", source_2)):
        (MIX / name).mkdir(parents=True, exist_ok=True)
        sf.write(MIX / name / f"{mixture_id}.wav", (audio / peak).astype(np.float32), SAMPLE_RATE)
    mixture_rows.append({
        "mixture_id": mixture_id, "language_1": left["language"], "language_2": right["language"],
        "overlap_ratio": overlap, "sir_db": sir_db,
        "audio": str(MIX / "mix" / f"{mixture_id}.wav"),
    })
mixtures = pd.DataFrame(mixture_rows)
mixtures.to_csv(MIX / "manifest.csv", index=False)
display(mixtures.head())

## Pyannote diarization

This uses the free gated Hugging Face `community-1` model locally in Colab. Accept the model and segmentation terms on Hugging Face before running it.

In [ ]:
from pyannote.audio import Pipeline

diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1", token=HF_TOKEN
)
diarization_rows = []
diarization_json = {}
for row in mixture_rows[:4]:
    try:
        annotation = diarization_pipeline(row["audio"])
        turns = []
        labels = set()
        for segment, _, speaker in annotation.itertracks(yield_label=True):
            labels.add(speaker)
            turns.append({"start": round(segment.start, 3), "end": round(segment.end, 3), "speaker": speaker})
        diarization_json[row["mixture_id"]] = turns
        diarization_rows.append({"mixture_id": row["mixture_id"], "speakers": len(labels),
                                 "turns": len(turns), "status": "ok"})
    except Exception as exc:
        diarization_rows.append({"mixture_id": row["mixture_id"], "speakers": np.nan,
                                 "turns": np.nan, "status": f"{type(exc).__name__}: {exc}"})
(RESULTS / "diarization.json").write_text(json.dumps(diarization_json, indent=2), encoding="utf-8")
diarization_table = pd.DataFrame(diarization_rows)
display(diarization_table)

## IndicWav2Vec / Sarvam transcription

IndicWav2Vec has published ULCA endpoints for some languages. For languages without a currently published endpoint, the notebook tries Sarvam if `SARVAM_API_KEY` is present; otherwise it records `unavailable` instead of fabricating text.

In [ ]:
INDICWAV2VEC_ENDPOINTS = {
    "hi": "https://ai4b-dev-asr.ulcacontrib.org/asr/v1/recognize/hi",
    "mr": "https://ai4b-dev-asr.ulcacontrib.org/asr/v1/recognize/mr",
    "ta": "https://ai4b-dev-asr.ulcacontrib.org/asr/v1/recognize/ta",
    "te": "https://ai4b-dev-asr.ulcacontrib.org/asr/v1/recognize/te",
}

def transcribe_sarvam(path, code):
    """Fallback transcription using Sarvam AI API if SARVAM_API_KEY is configured."""
    api_key = os.environ.get("SARVAM_API_KEY")
    if not api_key:
        return None
    try:
        url = "https://api.sarvam.ai/speech-to-text"
        headers = {"api-subscription-key": api_key}
        with open(path, "rb") as f:
            files = {"file": (Path(path).name, f, "audio/wav")}
            data = {"language_code": f"{code}-IN", "model": "saaras:v1"}
            resp = requests.post(url, headers=headers, files=files, data=data, timeout=60)
        if resp.status_code == 200:
            return {"status": "ok (sarvam)", "text": resp.json().get("transcript", "")}
    except Exception:
        pass
    return None

def indicwav2vec_transcribe(path, code):
    endpoint = INDICWAV2VEC_ENDPOINTS.get(code)
    if not endpoint:
        sarvam_res = transcribe_sarvam(path, code)
        if sarvam_res:
            return sarvam_res
        return {"status": "unavailable", "text": ""}
    payload = {
        "config": {"language": {"sourceLanguage": code},
                   "transcriptionFormat": {"value": "transcript"}, "audioFormat": "wav"},
        "audio": [{"audioContent": base64.b64encode(Path(path).read_bytes()).decode("ascii")}],
    }
    try:
        response = requests.post(endpoint, json=payload, timeout=60)
        response.raise_for_status()
        body = response.json()
        text = (body.get("output") or [{}])[0].get("source", "")
        return {"status": "ok", "text": text}
    except Exception as exc:
        sarvam_res = transcribe_sarvam(path, code)
        if sarvam_res:
            return sarvam_res
        return {"status": f"error: {type(exc).__name__}", "text": ""}

transcript_rows = []
for row in download_rows:
    result = indicwav2vec_transcribe(row["audio"], row["code"])
    transcript_rows.append({**row, **result})
transcripts = pd.DataFrame(transcript_rows)
transcripts.to_csv(RESULTS / "transcripts.csv", index=False)
display(transcripts[["language", "status", "text"]])


## Gemini cleanup and grounded report

Gemini is used only for cleanup, topic tags, and summarization. The prompt forbids adding facts that are absent from the raw transcript.

In [ ]:
def grounded_gemini_report(raw_text):
    if not os.environ.get("GEMINI_API_KEY"):
        return {"status": "unavailable", "summary": "", "tags": [], "dialogue": []}
    if not raw_text.strip():
        return {"status": "empty_input", "summary": "No transcription available for Gemini cleanup.", "tags": [], "dialogue": []}
    from google import genai
    import re
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    prompt = f"""
You are cleaning a noisy multilingual speech transcript.
Return valid JSON with keys: summary, tags, dialogue.
Use only information explicitly present in RAW TRANSCRIPT.
Do not invent names, numbers, locations, causes, or events.
Repair obvious ASR noise conservatively, preserve code-switching meaning, and identify topics only when supported.
RAW TRANSCRIPT:
{raw_text}
"""
    try:
        response = client.models.generate_content(model="gemini-2.0-flash", contents=prompt)
        text = response.text.strip()
        if "```" in text:
            match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
            if match:
                text = match.group(1)
            else:
                text = text.replace("```json", "").replace("```", "").strip()
        report = json.loads(text)
        report["status"] = "ok"
        return report
    except Exception as exc:
        return {"status": f"error: {type(exc).__name__}", "summary": "", "tags": [], "dialogue": []}

raw_text = "\n".join(
    f"{row['language']}: {row['text']}" for row in transcript_rows if row.get("text")
)
report = grounded_gemini_report(raw_text)
(RESULTS / "gemini_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
display(report)


## Metrics and plots

The plots report actual collection and diarization outcomes. Separation scoring is shown only when an estimated separated source is available; no placeholder accuracy is inserted.

In [ ]:
plot_rows = []
for row in mixture_rows:
    diar = next((item for item in diarization_rows if item["mixture_id"] == row["mixture_id"]), {})
    plot_rows.append({**row, "detected_speakers": diar.get("speakers", np.nan), "turns": diar.get("turns", np.nan)})
plot_table = pd.DataFrame(plot_rows)
plot_table.to_csv(RESULTS / "summary_metrics.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
if not plot_table.empty:
    sns.barplot(data=plot_table, x="mixture_id", y="overlap_ratio", ax=axes[0], color="#4EA8DE")
    axes[0].set_title("Requested overlap ratio")
    sns.barplot(data=plot_table, x="mixture_id", y="detected_speakers", ax=axes[1], color="#F4A259")
    axes[1].set_title("Pyannote speaker count")
    sns.scatterplot(data=plot_table, x="overlap_ratio", y="turns", hue="mixture_id", ax=axes[2])
    axes[2].set_title("Turns vs overlap")
for axis in axes:
    axis.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(RESULTS / "summary_plots.png", dpi=160)
display(plot_table)
print(f"All standalone artifacts are under {RESULTS}")